# Investment-Grade Real Estate Decision Model

This notebook is written in investment-committee format: data credibility, model robustness, downside risk, and allocation recommendation.

## 1) Data credibility and underwriting motivation
The framework is motivated by real allocation failures where base-case IRR looked attractive but correlated financing/rent shocks impaired equity outcomes.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from src.real_estate_investment_model import (
    build_hybrid_transaction_dataset, walk_forward_validation,
    default_investment_cases, monte_carlo_case, evaluate_investment_decision, DecisionPolicy
)
sns.set_theme(style='whitegrid', context='talk')
Path('../results/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
df = build_hybrid_transaction_dataset(n_properties_per_metro=160, seed=42)
df[['date','metro','sale_price','annual_rent','mortgage_rate','gdp_growth']].head()

In [ ]:
df.groupby('metro')[['sale_price','annual_rent','cap_rate']].median()

## 2) Walk-forward valuation reliability
Expanding-window folds test whether valuation signal survives changing macro regimes.

In [ ]:
model_summary, meta = walk_forward_validation(df, min_train_years=3, seed=42)
model_summary

In [ ]:
folds = meta['folds']
fig, ax = plt.subplots(figsize=(10,5))
sns.lineplot(data=folds, x='test_year', y='rmse', hue='model', marker='o', ax=ax)
ax.set_title('Walk-forward RMSE by Test Year')
fig.tight_layout(); fig.savefig('../results/figures/walk_forward_rmse.png', bbox_inches='tight')

## 3) Investment cases
Underwriting profiles represent realistic mandate buckets (Core, Value-Add, Opportunistic).

In [ ]:
cases = default_investment_cases()
pd.DataFrame([c.__dict__ for c in cases])[['name','purchase_price','ltv','initial_rate','annual_gross_rent','hurdle_irr']]

## 4) Correlated regime Monte Carlo
Jointly simulates price growth, rent shocks, and financing rates under Bull/Base/Bear probabilities.

In [ ]:
policy = DecisionPolicy()
rows = []
for c in cases:
    sim = monte_carlo_case(c, n_sims=12000, seed=41)
    rows.append({'case': c.name, **evaluate_investment_decision(sim, c.hurdle_irr, policy=policy)})
decision_df = pd.DataFrame(rows)
decision_df[['case','median_irr','median_npv','prob_npv_negative','prob_irr_below_hurdle','recommendation']]

In [ ]:
sim_case = monte_carlo_case(cases[1], n_sims=12000, seed=99)
fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.histplot(sim_case['npv'], bins=60, kde=True, ax=axes[0], color='#2a9d8f')
axes[0].axvline(sim_case['npv'].quantile(0.05), ls='--', c='black', label='VaR 5%')
axes[0].legend(); axes[0].set_title('NPV Distribution (Value-Add)')
sns.histplot(sim_case['irr'].dropna(), bins=60, kde=True, ax=axes[1], color='#e9c46a')
axes[1].axvline(cases[1].hurdle_irr, ls='--', c='red', label='Hurdle IRR')
axes[1].legend(); axes[1].set_title('IRR Distribution (Value-Add)')
fig.tight_layout(); fig.savefig('../results/figures/value_add_risk_distribution.png', bbox_inches='tight')

## 5) Key Investment Insights
1. Ranking deals by expected IRR alone misallocates risk capital; downside probabilities materially change priority.
2. Financing-rate persistence can dominate NOI improvements in leveraged structures.
3. Value-add strategies require execution-gated capital release due to convex downside in rent misses.
4. Exit-cap expansion is a major terminal-value risk, requiring conservative entry basis.
5. VaR/CVaR should drive position sizing and concentration limits, not just risk commentary.
6. Regime-aware approval conditions improve timing discipline and reduce avoidable drawdowns.

## 6) Why this demonstrates readiness for MIT MFin
- Integrates predictive modeling, financial engineering, and risk-governance logic.
- Translates quantitative outputs into institutional recommendation language.
- Shows judgment under uncertainty, not just model-building proficiency.